In [2]:
# Here we will rename files from Emrah's dataset to a more manageable format.
# Each block includes a different figure from the paper https://www.nature.com/articles/s41467-025-65892-9

# We will rename files in the simple format: 
# <figureno>_<replicate>.TIF>
# store the original filenames in a json file for reference.
# Format of json file:
# {source_filename:old_filename, target_filename: new_filename}
# This way we can easily map back to the original filenames if needed.

import os 
import json 
import shutil
import natsort 
target_dir = "/hpc/group/youlab/ks723/storage/Exp_images/EmrahPaKp_dataset_renamed"

if not os.path.exists(target_dir):
    os.makedirs(target_dir)

In [ ]:
# Supp Fig 5 30C

# Original filenames 
# {x}_{y}_{z}.TIF or {additional_info}_{x}_{y}_{z}.TIF
# desired filenames
# {number}_{replicate}.TIF
# we will group files by x value, rest will act as replicates
# note additional_info is a new grouping, will act as a new x value

source_dir= '/hpc/group/youlab/ks723/storage/Exp_images/41467_2025_65892_MOESM4_ESM/Source Data/SupFig5/toZenodo_SupFig5_30C'

groups={}
json_data={}

for file in natsort.natsorted(os.listdir(source_dir)):
    if file.endswith(".TIF") and not file.startswith("._"):  # ignore leftover of Icloud backups
        parts = file.split("_")
        if len(parts) == 3:
            x, y, z_ext = parts
            additional_info = None
            group_key =x
        elif len(parts) == 4:
            additional_info, x, y, z_ext = parts
            group_key = f"{additional_info}_{x}"
        else:
            continue  # unexpected format

        z = z_ext.split(".")[0]  # remove .TIF extension

        # store in groups dictionary 
        if group_key not in groups:
            groups[group_key] = []
        groups[group_key].append((y, z))

counter = 1  # to number the figures
for group_key, replicates in natsort.natsorted(groups.items()):
    
    for idx, (y, z) in enumerate(natsort.natsorted(replicates), start=1):
        source_filename = f"{group_key}_{y}_{z}.TIF" 
        target_filename = f"{counter}_{idx}.TIF"

        # copy and rename file
        shutil.copyfile(os.path.join(source_dir, source_filename), os.path.join(target_dir, target_filename))
        
        # store mapping in json
        json_data[target_filename] = source_filename

    counter+=1
      

# store json file 

with open(os.path.join(target_dir, "filename_mapping.json"), "w") as json_file:
    # json format is {source_filename: source_filename, target_filename: target_filename}
    for target_filename, source_filename in json_data.items():
        mapping = {"source_filename": source_filename, "target_filename": target_filename}
        json.dump(mapping, json_file)
        json_file.write("\n")  # write each mapping on a new line




In [7]:
# now we do this with other folders too
# we will update counter accordingly

# see length of json file counter
with open(os.path.join(target_dir, "filename_mapping.json"), "r") as f:
    lines = f.readlines()
    last_mapping = json.loads(lines[-1])
    counter = int(last_mapping["target_filename"].split("_")[0]) + 1  # increment last count by 1

# Supp Fig 5 40C

source_dir= '/hpc/group/youlab/ks723/storage/Exp_images/41467_2025_65892_MOESM4_ESM/Source Data/SupFig5/toZenodo_SupFig5_40C'

groups={}
json_data={}

for file in natsort.natsorted(os.listdir(source_dir)):
    if file.endswith(".TIF") and not file.startswith("._"):  # ignore leftover of Icloud backups
        parts = file.split("_")
        if len(parts) == 3:
            x, y, z_ext = parts
            additional_info = None
            group_key =x
        elif len(parts) == 4:
            additional_info, x, y, z_ext = parts
            group_key = f"{additional_info}_{x}"
        else:
            continue  # unexpected format

        z = z_ext.split(".")[0]  # remove .TIF extension

        # store in groups dictionary 
        if group_key not in groups:
            groups[group_key] = []
        groups[group_key].append((y, z))


for group_key, replicates in natsort.natsorted(groups.items()):
    
    for idx, (y, z) in enumerate(natsort.natsorted(replicates), start=1):
        source_filename = f"{group_key}_{y}_{z}.TIF" 
        target_filename = f"{counter}_{idx}.TIF"

        # copy and rename file
        shutil.copyfile(os.path.join(source_dir, source_filename), os.path.join(target_dir, target_filename))
        
        # store mapping in json
        json_data[target_filename] = source_filename

    counter+=1
      

# store json file 

with open(os.path.join(target_dir, "filename_mapping.json"), "a") as json_file:
    # json format is {source_filename: source_filename, target_filename: target_filename}
    for target_filename, source_filename in json_data.items():
        mapping = {"source_filename": source_filename, "target_filename": target_filename}
        json.dump(mapping, json_file)
        json_file.write("\n")  # write each mapping on a new line


In [7]:
# Supp Fig 7

# do some preprocess to make them in the same folders 
# replicate 2 
source_dir= '/hpc/group/youlab/ks723/storage/Exp_images/41467_2025_65892_MOESM4_ESM/Source Data/SupFig7_BiologicalReplicate2_20230802'
# combine CTX- and CTX+ into one folder 
# add CTX_ in prefix of CTX+ files 

renamed_dir = '/hpc/group/youlab/ks723/storage/Exp_images/41467_2025_65892_MOESM4_ESM/Source Data/SupFig7'
os.makedirs(renamed_dir, exist_ok=True)

# some more hardcoding here for now
# Read the filename, change {xy}.TIF to x_y.TIF,y is the last letter and a number. If y is not a number , keep x and rename like x_1.TIF
# also add an identifier like C2(biological clone 2) to distinguish from other biological replicates 
for file in os.listdir(os.path.join(source_dir, "CTX+")):
    if file.endswith(".TIF") and not file.startswith("._"):
        base, ext = os.path.splitext(file)
        x = base[:-1]
        y = base[-1]
        if y.isdigit():
            new_name = f"CTX_{x}_C2_{y}{ext}"
        else:
            new_name = f"CTX_{x}{y}_C2_1{ext}"
        shutil.copyfile(os.path.join(source_dir, "CTX+", file), os.path.join(renamed_dir, new_name))


# do same for CTX- files, here we have no prefix in front 

for file in os.listdir(os.path.join(source_dir, "CTX-")):
    if file.endswith(".TIF") and not file.startswith("._"):
        base, ext = os.path.splitext(file)
        x = base[:-1]
        y = base[-1]
        if y.isdigit():
            new_name = f"{x}_C2_{y}{ext}"
        else:
            new_name = f"{x}{y}_C2_1{ext}"
        shutil.copyfile(os.path.join(source_dir, "CTX-", file), os.path.join(renamed_dir, new_name))


# replicate 3

source_dir= '/hpc/group/youlab/ks723/storage/Exp_images/41467_2025_65892_MOESM4_ESM/Source Data/SupFig7_BiologicalReplicate3_20230815'
# combine CTX- and CTX+ into one folder 
# add CTX_ in prefix of CTX+ files 

renamed_dir = '/hpc/group/youlab/ks723/storage/Exp_images/41467_2025_65892_MOESM4_ESM/Source Data/SupFig7'

# some more hardcoding here for now
# Read the filename, change {xy}.TIF to x_y.TIF,y is the last letter and a number. If y is not a number , keep x and rename like x_1.TIF
# also add an identifier like C3(biological clone 3) to distinguish from other biological replicates 
for file in os.listdir(os.path.join(source_dir, "CTX+")):
    if file.endswith(".TIF") and not file.startswith("._"):
        base, ext = os.path.splitext(file)
        x = base[:-1]
        y = base[-1]
        if y.isdigit():
            new_name = f"CTX_{x}_C3_{y}{ext}"
        else:
            new_name = f"CTX_{x}{y}_C3_1{ext}"
        shutil.copyfile(os.path.join(source_dir, "CTX+", file), os.path.join(renamed_dir, new_name))


# do same for CTX- files, here we have no prefix in front 

for file in os.listdir(os.path.join(source_dir, "CTX-")):
    if file.endswith(".TIF") and not file.startswith("._"):
        base, ext = os.path.splitext(file)
        x = base[:-1]
        y = base[-1]
        if y.isdigit():
            new_name = f"{x}_C3_{y}{ext}"
        else:
            new_name = f"{x}{y}_C3_1{ext}"
        shutil.copyfile(os.path.join(source_dir, "CTX-", file), os.path.join(renamed_dir, new_name))

# replicate 1
# already in same folder
# just remove NoCefo from the filenames which have them at the prefix 
# also change Cefo2p5 to CTX for consistency 
# also add identifier like C1 (biological clone 1) to distinguish from other biological replicates

source_dir= '/hpc/group/youlab/ks723/storage/Exp_images/41467_2025_65892_MOESM4_ESM/Source Data/SupFig7_BiologicalReplicate1_20220906'

for file in os.listdir(source_dir):
    if file.endswith(".TIF") and not file.startswith("._"):

        # all files are x_y_z.TIF
        # just adding C1 after y first

        # split file into parts based on underscores
        parts = file.split("_")
        parts[1]= parts[1]+"_C1"  # add _C1 to second part
        new_name = "_".join(parts)  # join back to form new filename

        if new_name.startswith("NoCefo_"):
            new_name = new_name.replace("NoCefo_", "")
        else:
            new_name = new_name.replace("Cefo2p5", "CTX")

        # remove Mix from name if present
        if "Mix" in new_name:
            new_name = new_name.replace("Mix", "")

        # remove 0s from the names
        new_name = new_name.replace("0", "")

        shutil.copyfile(os.path.join(source_dir, file), os.path.join(renamed_dir, new_name))


# also final for simplicity 
# change Mix/mix to PaKp
# change p to P, change k to K

for file in os.listdir(renamed_dir):
    if file.endswith(".TIF") and not file.startswith("._"):
        new_name = file

        if "Mix" in new_name:
            new_name = new_name.replace("Mix", "PaKp")
        if "mix" in new_name:
            new_name = new_name.replace("mix", "PaKp")

        new_name = new_name.replace("p", "P")
        new_name = new_name.replace("k", "K")

        os.rename(os.path.join(renamed_dir, file), os.path.join(renamed_dir, new_name))



In [8]:
# now we have all files in the same folder with consistent namings
# now we will rename them to figure_replicate.TIF format

# see length of json file counter
with open(os.path.join(target_dir, "filename_mapping.json"), "r") as f:
    lines = f.readlines()
    last_mapping = json.loads(lines[-1])
    counter = int(last_mapping["target_filename"].split("_")[0]) + 1  # increment last count by 1

    
# Supp Fig 7

source_dir= renamed_dir

groups={}
json_data={}

for file in natsort.natsorted(os.listdir(source_dir)):
    if file.endswith(".TIF") and not file.startswith("._"):  # ignore leftover of Icloud backups
        parts = file.split("_")
        if len(parts) == 3:
            x, y, z_ext = parts
            additional_info = None
            group_key =x
        elif len(parts) == 4:
            additional_info, x, y, z_ext = parts
            group_key = f"{additional_info}_{x}"
        else:
            continue  # unexpected format

        z = z_ext.split(".")[0]  # remove .TIF extension

        # store in groups dictionary 
        if group_key not in groups:
            groups[group_key] = []
        groups[group_key].append((y, z))


for group_key, replicates in natsort.natsorted(groups.items()):
    
    for idx, (y, z) in enumerate(natsort.natsorted(replicates), start=1):
        source_filename = f"{group_key}_{y}_{z}.TIF" 
        target_filename = f"{counter}_{idx}.TIF"

        # copy and rename file
        shutil.copyfile(os.path.join(source_dir, source_filename), os.path.join(target_dir, target_filename))
        
        # store mapping in json
        json_data[target_filename] = source_filename

    counter+=1
      

# store json file 

with open(os.path.join(target_dir, "filename_mapping.json"), "a") as json_file:
    # json format is {source_filename: source_filename, target_filename: target_filename}
    for target_filename, source_filename in json_data.items():
        mapping = {"source_filename": source_filename, "target_filename": target_filename}
        json.dump(mapping, json_file)
        json_file.write("\n")  # write each mapping on a new line



In [10]:
# Supp Fig 9 

# folder replicate 2 
# has files like CTX{conc}_{biologicalreplicate}_{techreplicate}.TIF
# We will just rename to CTX_{conc}_{biologicalreplicate}_{techreplicate}.TIF

source_dir= '/hpc/group/youlab/ks723/storage/Exp_images/41467_2025_65892_MOESM4_ESM/Source Data/SupFig9/BiologicalReplicate2_20230926'
renamed_dir = '/hpc/group/youlab/ks723/storage/Exp_images/41467_2025_65892_MOESM4_ESM/Source Data/SupFig9/Aggregated'

os.makedirs(renamed_dir, exist_ok=True)

for file in os.listdir(source_dir):
    if file.endswith(".TIF") and not file.startswith("._"):
        
        file_parts = file.split("CTX")
        if len(file_parts) == 2:
            new_name = "CTX_" + file_parts[1]
            shutil.copyfile(os.path.join(source_dir, file), os.path.join(renamed_dir, new_name))

# folder replicate 1
# has files like CTX{conc}_{techreplicate}.TIF
# We will just rename to CTX_{conc}_C03_{techreplicate}.TIF 

source_dir ='/hpc/group/youlab/ks723/storage/Exp_images/41467_2025_65892_MOESM4_ESM/Source Data/SupFig9/BiologicalReplicate1_20230627'

for file in os.listdir(source_dir):
    if file.endswith(".TIF") and not file.startswith("._"):
        
        file_parts = file.split("CTX")
        if len(file_parts) == 2:
            new_name = "CTX_" + file_parts[1].replace("_", "_C03_")
            shutil.copyfile(os.path.join(source_dir, file), os.path.join(renamed_dir, new_name))

# for consistency change all names to caps 

for file in os.listdir(renamed_dir):
    if file.endswith(".TIF") and not file.startswith("._"):
        old_path = os.path.join(renamed_dir, file)
        new_name = file.upper()
        new_path = os.path.join(renamed_dir, new_name)
        os.rename(old_path, new_path)

In [11]:
# now we will rename them to figure_replicate.TIF format
# see length of json file counter

# see length of json file counter
with open(os.path.join(target_dir, "filename_mapping.json"), "r") as f:
    lines = f.readlines()
    last_mapping = json.loads(lines[-1])
    counter = int(last_mapping["target_filename"].split("_")[0]) + 1  # increment last count by 1

    
# Supp Fig 9

source_dir= renamed_dir

groups={}
json_data={}

for file in natsort.natsorted(os.listdir(source_dir)):
    if file.endswith(".TIF") and not file.startswith("._"):  # ignore leftover of Icloud backups
        parts = file.split("_")
        if len(parts) == 3:
            x, y, z_ext = parts
            additional_info = None
            group_key =x
        elif len(parts) == 4:
            additional_info, x, y, z_ext = parts
            group_key = f"{additional_info}_{x}"
        else:
            continue  # unexpected format

        z = z_ext.split(".")[0]  # remove .TIF extension

        # store in groups dictionary 
        if group_key not in groups:
            groups[group_key] = []
        groups[group_key].append((y, z))


for group_key, replicates in natsort.natsorted(groups.items()):
    
    for idx, (y, z) in enumerate(natsort.natsorted(replicates), start=1):
        source_filename = f"{group_key}_{y}_{z}.TIF" 
        target_filename = f"{counter}_{idx}.TIF"

        # copy and rename file
        shutil.copyfile(os.path.join(source_dir, source_filename), os.path.join(target_dir, target_filename))
        
        # store mapping in json
        json_data[target_filename] = source_filename

    counter+=1
      

# store json file 

with open(os.path.join(target_dir, "filename_mapping.json"), "a") as json_file:
    # json format is {source_filename: source_filename, target_filename: target_filename}
    for target_filename, source_filename in json_data.items():
        mapping = {"source_filename": source_filename, "target_filename": target_filename}
        json.dump(mapping, json_file)
        json_file.write("\n")  # write each mapping on a new line



In [ ]:
# Supp Fig 13

# rename files in some of the folders first to have consistent namings
# first 1x replicates 

source_dir = '/hpc/group/youlab/ks723/storage/Exp_images/41467_2025_65892_MOESM4_ESM/Source Data/SupFig13/IndependentReplicate01/1x_Klebsiella_seeding_density'
renamed_dir = '/hpc/group/youlab/ks723/storage/Exp_images/41467_2025_65892_MOESM4_ESM/Source Data/SupFig13/Aggregated'

os.makedirs(renamed_dir, exist_ok=True)

# file is d{distance}_{distancelabel}.TIF 
# one file has {distance_label} missing, we will use the d0 distance for this
# to rename like 1_d{distancelabel}_C1_1.TIF 
# distance is like 01,02 etc, we will remove 0
# replicate number and tech replicate number are both 1 here

for file in os.listdir(source_dir):
    if file.endswith(".TIF") and not file.startswith("._"):
        base, ext = os.path.splitext(file)
        parts = base.split("_")
        if len(parts) == 2:
            distance_part, distance_label = parts
        else:
            distance_part = parts[0]
            distance_label = "0"  # default to d0 if missing

        # take only last character from distance label
        distance_label = distance_label[-1]
        new_name = f"1_d{distance_label}_C1_1{ext}"
        shutil.copyfile(os.path.join(source_dir, file), os.path.join(renamed_dir, new_name))


# now with second replicate 1x folder
source_dir = '/hpc/group/youlab/ks723/storage/Exp_images/41467_2025_65892_MOESM4_ESM/Source Data/SupFig13/IndependentReplicate02/1x_Klebsiella_seeding_density'

# files here are like CTX2p5_d{xy}_{replicate}.TIF where xy is two digit distance label 
# we will rename to 1_d{y}_C2_{replicate}.TIF

for file in os.listdir(source_dir):
    if file.endswith(".TIF") and not file.startswith("._"):
        base, ext = os.path.splitext(file)
        parts = base.split("_")
        distance_label = parts[1]  # CTX2p5_d{xy}
        replicate = parts[2]  # replicate number

        # extract distance label
        distance_label = distance_label[-1] # take only last character

        new_name = f"1_d{distance_label}_C2_{replicate}{ext}"
        shutil.copyfile(os.path.join(source_dir, file), os.path.join(renamed_dir, new_name))

# now with third replicate 1x folder
source_dir = '/hpc/group/youlab/ks723/storage/Exp_images/41467_2025_65892_MOESM4_ESM/Source Data/SupFig13/IndependentReplicate03/1x_Klebsiella_seeding_density'

# files here are like d{distancelabel}_{x}_{y}_{replicate}.TIF
# we will rename to 1_d{distancelabel}_C3_{replicate}.TIF
# distance label here is 1 digit


for file in os.listdir(source_dir): 
    if file.endswith(".TIF") and not file.startswith("._"):
        base, ext = os.path.splitext(file)
        parts = base.split("_")
        distance_part = parts[0]  # d{distancelabel}
        replicate = parts[3]  # replicate number

        distance_label = distance_part[-1]  # take only last character, since including d later explicitly

        new_name = f"1_d{distance_label}_C3_{replicate}{ext}"
        shutil.copyfile(os.path.join(source_dir, file), os.path.join(renamed_dir, new_name))

# now fourth replicate 1x folder
# these are experiments I did I think, so mostly labels are consistent
# files are like 1_d{distancelabel}_{replicate}.TIF
# we will rename to 1_d{distancelabel}_C4_{replicate}.TIF

source_dir = '/hpc/group/youlab/ks723/storage/Exp_images/41467_2025_65892_MOESM4_ESM/Source Data/SupFig13/IndependentReplicate04/1x_Klebsiella_seeding_density'

for file in os.listdir(source_dir):
    if file.endswith(".TIF") and not file.startswith("._"):
        base, ext = os.path.splitext(file)
        parts = base.split("_")
        condition= parts[0]  # 1
        distance_part = parts[1]  # d{distancelabel}
        replicate = parts[2]  # replicate number

        distance_label = distance_part[-1]  # take only last character, since including d later explicitly

        new_name = f"{condition}_d{distance_label}_C4_{replicate}{ext}"
        shutil.copyfile(os.path.join(source_dir, file), os.path.join(renamed_dir, new_name))


# fifth replicate folder 
# same shit

source_dir = '/hpc/group/youlab/ks723/storage/Exp_images/41467_2025_65892_MOESM4_ESM/Source Data/SupFig13/IndependentReplicate05/1x_Klebsiella_seeding_density'

for file in os.listdir(source_dir):
    if file.endswith(".TIF") and not file.startswith("._"):
        base, ext = os.path.splitext(file)
        parts = base.split("_")
        condition= parts[0]  # 1
        distance_part = parts[1]  # d{distancelabel}
        replicate = parts[2]  # replicate number

        distance_label = distance_part[-1]  # take only last character, since including d later explicitly

        new_name = f"{condition}_d{distance_label}_C5_{replicate}{ext}"
        shutil.copyfile(os.path.join(source_dir, file), os.path.join(renamed_dir, new_name))

# sixth replicate
# here we go again

source_dir = '/hpc/group/youlab/ks723/storage/Exp_images/41467_2025_65892_MOESM4_ESM/Source Data/SupFig13/IndependentReplicate06/1x_Klebsiella_seeding_density'

for file in os.listdir(source_dir):
    if file.endswith(".TIF") and not file.startswith("._"):
        base, ext = os.path.splitext(file)
        parts = base.split("_")
        condition= parts[0]  # 1
        distance_part = parts[1]  # d{distancelabel}
        replicate = parts[2]  # replicate number

        distance_label = distance_part[-1]  # take only last character, since including d later explicitly

        new_name = f"{condition}_d{distance_label}_C6_{replicate}{ext}"
        shutil.copyfile(os.path.join(source_dir, file), os.path.join(renamed_dir, new_name))




In [14]:
# now for 0.1x folders
# first replicate 0.1x
#  structure similar to 1x replicate 3 
# we just add point1 intead of 1 in the beginning
# even though this is first condition here, we still keep 3 as biological replicate number to be consistent

source_dir = '/hpc/group/youlab/ks723/storage/Exp_images/41467_2025_65892_MOESM4_ESM/Source Data/SupFig13/IndependentReplicate03/0p1x_Klebsiella_seeding_density' 

for file in os.listdir(source_dir): 
    if file.endswith(".TIF") and not file.startswith("._"):
        base, ext = os.path.splitext(file)
        parts = base.split("_")
        distance_part = parts[0]  # d{distancelabel}
        replicate = parts[3]  # replicate number

        distance_label = distance_part[-1]  # take only last character, since including d later explicitly

        new_name = f"point1_d{distance_label}_C3_{replicate}{ext}"
        shutil.copyfile(os.path.join(source_dir, file), os.path.join(renamed_dir, new_name))

# second replicate 0.1x
# structure similar to 1x replicate 4
# we just add point1 intead of 1 in the beginning

source_dir = '/hpc/group/youlab/ks723/storage/Exp_images/41467_2025_65892_MOESM4_ESM/Source Data/SupFig13/IndependentReplicate04/0p1x_Klebsiella_seeding_density'

for file in os.listdir(source_dir):
    if file.endswith(".TIF") and not file.startswith("._"):
        base, ext = os.path.splitext(file)
        parts = base.split("_")
        condition= parts[0]  # point1
        distance_part = parts[1]  # d{distancelabel}
        replicate = parts[2]  # replicate number

        distance_label = distance_part[-1]  # take only last character, since including d later explicitly

        new_name = f"{condition}_d{distance_label}_C4_{replicate}{ext}"
        shutil.copyfile(os.path.join(source_dir, file), os.path.join(renamed_dir, new_name))

# third replicate 0.1x
# same as above

source_dir= '/hpc/group/youlab/ks723/storage/Exp_images/41467_2025_65892_MOESM4_ESM/Source Data/SupFig13/IndependentReplicate05/0p1x_Klebsiella_seeding_density'

for file in os.listdir(source_dir):
    if file.endswith(".TIF") and not file.startswith("._"):
        base, ext = os.path.splitext(file)
        parts = base.split("_")
        condition= parts[0]  # point1
        distance_part = parts[1]  # d{distancelabel}
        replicate = parts[2]  # replicate number

        distance_label = distance_part[-1]  # take only last character, since including d later explicitly

        new_name = f"{condition}_d{distance_label}_C5_{replicate}{ext}"
        shutil.copyfile(os.path.join(source_dir, file), os.path.join(renamed_dir, new_name))


# fourth replicate 0.1x
# same as above 

source_dir = '/hpc/group/youlab/ks723/storage/Exp_images/41467_2025_65892_MOESM4_ESM/Source Data/SupFig13/IndependentReplicate06/0p1x_Klebsiella_seeding_density'

for file in os.listdir(source_dir):
    if file.endswith(".TIF") and not file.startswith("._"):
        base, ext = os.path.splitext(file)
        parts = base.split("_")
        condition= parts[0]  # point1
        distance_part = parts[1]  # d{distancelabel}
        replicate = parts[2]  # replicate number

        distance_label = distance_part[-1]  # take only last character, since including d later explicitly

        new_name = f"{condition}_d{distance_label}_C6_{replicate}{ext}"
        shutil.copyfile(os.path.join(source_dir, file), os.path.join(renamed_dir, new_name))

# fifth replicate 0.1x 
# same as above

source_dir = '/hpc/group/youlab/ks723/storage/Exp_images/41467_2025_65892_MOESM4_ESM/Source Data/SupFig13/IndependentReplicate07/0p1x_Klebsiella_seeding_density'

for file in os.listdir(source_dir):
    if file.endswith(".TIF") and not file.startswith("._"):
        base, ext = os.path.splitext(file)
        parts = base.split("_")
        condition= parts[0]  # point1
        distance_part = parts[1]  # d{distancelabel}
        replicate = parts[2]  # replicate number

        distance_label = distance_part[-1]  # take only last character, since including d later explicitly

        new_name = f"{condition}_d{distance_label}_C7_{replicate}{ext}"
        shutil.copyfile(os.path.join(source_dir, file), os.path.join(renamed_dir, new_name))

        

In [3]:
# now for 10x 

# first replicate 10x
# structure similar to 1x replicate 3
# just add 10 in the beginning
# again we keep C3 as biological replicate number for consistency

renamed_dir = '/hpc/group/youlab/ks723/storage/Exp_images/41467_2025_65892_MOESM4_ESM/Source Data/SupFig13/Aggregated'
source_dir = '/hpc/group/youlab/ks723/storage/Exp_images/41467_2025_65892_MOESM4_ESM/Source Data/SupFig13/IndependentReplicate03/10x_Klebsiella_seeding_density' 

for file in os.listdir(source_dir):
    if file.endswith(".TIF") and not file.startswith("._"):
        base, ext = os.path.splitext(file)
        parts = base.split("_")
        distance_part = parts[0]  # d{distancelabel}
        replicate = parts[3]  # replicate number

        distance_label = distance_part[-1]  # take only last character, since including d later explicitly

        new_name = f"10_d{distance_label}_C3_{replicate}{ext}"
        shutil.copyfile(os.path.join(source_dir, file), os.path.join(renamed_dir, new_name))


# second replicate 10x 
# structure similar to 1x replicate 4

source_dir = '/hpc/group/youlab/ks723/storage/Exp_images/41467_2025_65892_MOESM4_ESM/Source Data/SupFig13/IndependentReplicate04/10x_Klebsiella_seeding_density'

for file in os.listdir(source_dir):
    if file.endswith(".TIF") and not file.startswith("._"):
        base, ext = os.path.splitext(file)
        parts = base.split("_")
        condition= parts[0]  # 10
        distance_part = parts[1]  # d{distancelabel}
        replicate = parts[2]  # replicate number

        distance_label = distance_part[-1]  # take only last character, since including d later explicitly

        new_name = f"{condition}_d{distance_label}_C4_{replicate}{ext}"
        shutil.copyfile(os.path.join(source_dir, file), os.path.join(renamed_dir, new_name))


# third replicate 10x 
# structure similar to 1x replicate 6 

source_dir = '/hpc/group/youlab/ks723/storage/Exp_images/41467_2025_65892_MOESM4_ESM/Source Data/SupFig13/IndependentReplicate06/10x_Klebsiella_seeding_density'   

for file in os.listdir(source_dir):
    if file.endswith(".TIF") and not file.startswith("._"):
        base, ext = os.path.splitext(file)
        parts = base.split("_")
        condition= parts[0]  # 10
        distance_part = parts[1]  # d{distancelabel}
        replicate = parts[2]  # replicate number

        distance_label = distance_part[-1]  # take only last character, since including d later explicitly

        new_name = f"{condition}_d{distance_label}_C6_{replicate}{ext}"
        shutil.copyfile(os.path.join(source_dir, file), os.path.join(renamed_dir, new_name))


In [4]:
# 0.5x 

# first replicate 0.5x
# structure similar to 1x replicate 4

source_dir = '/hpc/group/youlab/ks723/storage/Exp_images/41467_2025_65892_MOESM4_ESM/Source Data/SupFig13/IndependentReplicate04/0p5x_Klebsiella_seeding_density'

for file in os.listdir(source_dir):
    if file.endswith(".TIF") and not file.startswith("._"):
        base, ext = os.path.splitext(file)
        parts = base.split("_")
        condition= parts[0]  # point5
        distance_part = parts[1]  # d{distancelabel}
        replicate = parts[2]  # replicate number

        distance_label = distance_part[-1]  # take only last character, since including d later explicitly

        new_name = f"{condition}_d{distance_label}_C4_{replicate}{ext}"
        shutil.copyfile(os.path.join(source_dir, file), os.path.join(renamed_dir, new_name))


# second replicate 0.5x
# structure similar to 1x replicate 7

source_dir = '/hpc/group/youlab/ks723/storage/Exp_images/41467_2025_65892_MOESM4_ESM/Source Data/SupFig13/IndependentReplicate07/0p5x_Klebsiella_seeding_density'

for file in os.listdir(source_dir):
    if file.endswith(".TIF") and not file.startswith("._"):
        base, ext = os.path.splitext(file)
        parts = base.split("_")
        condition= parts[0]  # point5
        distance_part = parts[1]  # d{distancelabel}
        replicate = parts[2]  # replicate number

        distance_label = distance_part[-1]  # take only last character, since including d later explicitly

        new_name = f"{condition}_d{distance_label}_C7_{replicate}{ext}"
        shutil.copyfile(os.path.join(source_dir, file), os.path.join(renamed_dir, new_name))



In [ ]:
# 2.5x 

# first and only replicate 2.5x
# structure similar to 1x replicate 4

source_dir = '/hpc/group/youlab/ks723/storage/Exp_images/41467_2025_65892_MOESM4_ESM/Source Data/SupFig13/IndependentReplicate04/2p5x_Klebsiella_seeding_density'

for file in os.listdir(source_dir):
    if file.endswith(".TIF") and not file.startswith("._"):
        base, ext = os.path.splitext(file)
        parts = base.split("_")
        condition= parts[0]  # 2point5
        distance_part = parts[1]  # d{distancelabel}
        replicate = parts[2]  # replicate number

        distance_label = distance_part[-1]  # take only last character, since including d later explicitly

        new_name = f"{condition}_d{distance_label}_C4_{replicate}{ext}"
        shutil.copyfile(os.path.join(source_dir, file), os.path.join(renamed_dir, new_name))



In [ ]:
# 5x 

# first and only replicate 5x
# structure similar to 1x replicate 4

source_dir = '/hpc/group/youlab/ks723/storage/Exp_images/41467_2025_65892_MOESM4_ESM/Source Data/SupFig13/IndependentReplicate04/5x_Klebsiella_seeding_density'

for file in os.listdir(source_dir): 
    if file.endswith(".TIF") and not file.startswith("._"):
        base, ext = os.path.splitext(file)
        parts = base.split("_")
        condition= parts[0]  # 5x
        distance_part = parts[1]  # d{distancelabel}
        replicate = parts[2]  # replicate number

        distance_label = distance_part[-1]  # take only last character, since including d later explicitly

        new_name = f"{condition}_d{distance_label}_C4_{replicate}{ext}"
        shutil.copyfile(os.path.join(source_dir, file), os.path.join(renamed_dir, new_name))


In [7]:
# now we will rename them to figure_replicate.TIF format
# see length of json file counter

# see length of json file counter
with open(os.path.join(target_dir, "filename_mapping.json"), "r") as f:
    lines = f.readlines()
    last_mapping = json.loads(lines[-1])
    counter = int(last_mapping["target_filename"].split("_")[0]) + 1  # increment last count by 1

    
# Supp Fig 13

source_dir= renamed_dir

groups={}
json_data={}

for file in natsort.natsorted(os.listdir(source_dir)):
    if file.endswith(".TIF") and not file.startswith("._"):  # ignore leftover of Icloud backups
        parts = file.split("_")
        if len(parts) == 3:
            x, y, z_ext = parts
            additional_info = None
            group_key =x
        elif len(parts) == 4:
            additional_info, x, y, z_ext = parts
            group_key = f"{additional_info}_{x}"
        else:
            continue  # unexpected format

        z = z_ext.split(".")[0]  # remove .TIF extension

        # store in groups dictionary 
        if group_key not in groups:
            groups[group_key] = []
        groups[group_key].append((y, z))


for group_key, replicates in natsort.natsorted(groups.items()):
    
    for idx, (y, z) in enumerate(natsort.natsorted(replicates), start=1):
        source_filename = f"{group_key}_{y}_{z}.TIF" 
        target_filename = f"{counter}_{idx}.TIF"

        # copy and rename file
        shutil.copyfile(os.path.join(source_dir, source_filename), os.path.join(target_dir, target_filename))
        
        # store mapping in json
        json_data[target_filename] = source_filename

    counter+=1
      

# store json file 

with open(os.path.join(target_dir, "filename_mapping.json"), "a") as json_file:
    # json format is {source_filename: source_filename, target_filename: target_filename}
    for target_filename, source_filename in json_data.items():
        mapping = {"source_filename": source_filename, "target_filename": target_filename}
        json.dump(mapping, json_file)
        json_file.write("\n")  # write each mapping on a new line



In [8]:
# Supp Fig 18
# these files only contain one biological replicate 
# apart from these they are labelled consistently already
# we will just change the logic to two parts separated by underscore

renamed_dir = '/hpc/group/youlab/ks723/storage/Exp_images/41467_2025_65892_MOESM4_ESM/Source Data/SupFig18' 


# now we will rename them to figure_replicate.TIF format
# see length of json file counter

# see length of json file counter
with open(os.path.join(target_dir, "filename_mapping.json"), "r") as f:
    lines = f.readlines()
    last_mapping = json.loads(lines[-1])
    counter = int(last_mapping["target_filename"].split("_")[0]) + 1  # increment last count by 1

    
# Supp Fig 18

source_dir= renamed_dir

groups={}
json_data={}

for file in natsort.natsorted(os.listdir(source_dir)):
    if file.endswith(".TIF") and not file.startswith("._"):  # ignore leftover of Icloud backups
        parts = file.split("_")
        if len(parts) == 2:
            x, z_ext = parts
            additional_info = None
            group_key =x

        else:
            continue  # unexpected format

        z = z_ext.split(".")[0]  # remove .TIF extension

        # store in groups dictionary 
        if group_key not in groups:
            groups[group_key] = []
        groups[group_key].append(z)


for group_key, replicates in natsort.natsorted(groups.items()):
    
    for idx, z in enumerate(natsort.natsorted(replicates), start=1):
        source_filename = f"{group_key}_{z}.TIF" 
        target_filename = f"{counter}_{idx}.TIF"

        # copy and rename file
        shutil.copyfile(os.path.join(source_dir, source_filename), os.path.join(target_dir, target_filename))
        
        # store mapping in json
        json_data[target_filename] = source_filename

    counter+=1
      

# store json file 

with open(os.path.join(target_dir, "filename_mapping.json"), "a") as json_file:
    # json format is {source_filename: source_filename, target_filename: target_filename}
    for target_filename, source_filename in json_data.items():
        mapping = {"source_filename": source_filename, "target_filename": target_filename}
        json.dump(mapping, json_file)
        json_file.write("\n")  # write each mapping on a new line




In [10]:
# Supp Fig 19 

# Here files are placed in two folders for two biological replicates

# first replicate folder 
# Here we need to multiple stuff
# 1. Change 55_57 to PaBp 
# 2. Change 55 to Pa
# 3. Change 57 to Bp
# 4. Add C1 for biological replicate 1

source_dir = '/hpc/group/youlab/ks723/storage/Exp_images/41467_2025_65892_MOESM4_ESM/Source Data/SupFig19/BiologicalReplicate1_20240506'

renamed_dir = '/hpc/group/youlab/ks723/storage/Exp_images/41467_2025_65892_MOESM4_ESM/Source Data/SupFig19/Aggregated'
os.makedirs(renamed_dir, exist_ok=True)

for file in os.listdir(source_dir):
    if file.endswith(".TIF") and not file.startswith("._"):
        new_name = file

        if "55_57" in new_name:
            new_name = new_name.replace("55_57", "BpPa")
        elif "55" in new_name:
            new_name = new_name.replace("55", "Pa")
        elif "57" in new_name:
            new_name = new_name.replace("57", "Bp")

        # seperate to underscore parts 
        parts = new_name.split("_")
        
        if len(parts) == 3:
            additional_info, x, z_ext = parts
            new_name = f"{additional_info}_{x}_C1_{z_ext}"
        elif len(parts) == 2:
            x, z_ext = parts
            new_name = f"{x}_C1_{z_ext}"
        else:
            continue  # unexpected format

        shutil.copyfile(os.path.join(source_dir, file), os.path.join(renamed_dir, new_name))


# second replicate folder
# files here are of format {unneeded}_{x}_{additional_info}_{z}.TIF
# 1. We will remove unneeded part
# 2. We will add shift additional_info to the front if it exists
# 3. We will add C2 for biological replicate 2 after x 

source_dir = '/hpc/group/youlab/ks723/storage/Exp_images/41467_2025_65892_MOESM4_ESM/Source Data/SupFig19/BiologicalReplicate2_20240702'

for file in os.listdir(source_dir):
    if file.endswith(".TIF") and not file.startswith("._"):
        new_name = file

        # seperate to underscore parts 
        parts = new_name.split("_")
        
        if len(parts) == 4:
            unneeded, x, additional_info, z_ext = parts
            new_name = f"{additional_info}_{x}_C2_{z_ext}"
        elif len(parts) == 3:
            unneeded, x, z_ext = parts
            new_name = f"{x}_C2_{z_ext}"
        else:
            continue  # unexpected format

        shutil.copyfile(os.path.join(source_dir, file), os.path.join(renamed_dir, new_name))

    

In [11]:
# now we will rename them to figure_replicate.TIF format
# see length of json file counter

# see length of json file counter
with open(os.path.join(target_dir, "filename_mapping.json"), "r") as f:
    lines = f.readlines()
    last_mapping = json.loads(lines[-1])
    counter = int(last_mapping["target_filename"].split("_")[0]) + 1  # increment last count by 1

    
# Supp Fig 19

source_dir= renamed_dir

groups={}
json_data={}

for file in natsort.natsorted(os.listdir(source_dir)):
    if file.endswith(".TIF") and not file.startswith("._"):  # ignore leftover of Icloud backups
        parts = file.split("_")
        if len(parts) == 3:
            x, y, z_ext = parts
            additional_info = None
            group_key =x
        elif len(parts) == 4:
            additional_info, x, y, z_ext = parts
            group_key = f"{additional_info}_{x}"
        else:
            continue  # unexpected format

        z = z_ext.split(".")[0]  # remove .TIF extension

        # store in groups dictionary 
        if group_key not in groups:
            groups[group_key] = []
        groups[group_key].append((y, z))


for group_key, replicates in natsort.natsorted(groups.items()):
    
    for idx, (y, z) in enumerate(natsort.natsorted(replicates), start=1):
        source_filename = f"{group_key}_{y}_{z}.TIF" 
        target_filename = f"{counter}_{idx}.TIF"

        # copy and rename file
        shutil.copyfile(os.path.join(source_dir, source_filename), os.path.join(target_dir, target_filename))
        
        # store mapping in json
        json_data[target_filename] = source_filename

    counter+=1
      

# store json file 

with open(os.path.join(target_dir, "filename_mapping.json"), "a") as json_file:
    # json format is {source_filename: source_filename, target_filename: target_filename}
    for target_filename, source_filename in json_data.items():
        mapping = {"source_filename": source_filename, "target_filename": target_filename}
        json.dump(mapping, json_file)
        json_file.write("\n")  # write each mapping on a new line



In [12]:
# Supp Fig 20:
# Structure is similar to Supp Fig 5
# But we will do a small change, adding temp first to distinguish and aggregate to a single folder 

renamed_dir = '/hpc/group/youlab/ks723/storage/Exp_images/41467_2025_65892_MOESM4_ESM/Source Data/SupFig20/Aggregated'
os.makedirs(renamed_dir, exist_ok=True)

# first folder , 30C
source_dir = '/hpc/group/youlab/ks723/storage/Exp_images/41467_2025_65892_MOESM4_ESM/Source Data/SupFig20/30C'

for file in os.listdir(source_dir):
    if file.endswith(".TIF") and not file.startswith("._"):
        
        parts = file.split("_")
        if len(parts) == 3:
            x, y, z_ext = parts
            new_name = f"{x}30C_{y}_{z_ext}"
        elif len(parts) == 4:
            additional_info, x, y, z_ext = parts
            new_name = f"{additional_info}_{x}30C_{y}_{z_ext}"
        else:   
            continue  # unexpected format
  
        shutil.copyfile(os.path.join(source_dir, file), os.path.join(renamed_dir, new_name))

# second folder 40 C

source_dir = '/hpc/group/youlab/ks723/storage/Exp_images/41467_2025_65892_MOESM4_ESM/Source Data/SupFig20/40C'  

for file in os.listdir(source_dir):
    if file.endswith(".TIF") and not file.startswith("._"):
        
        parts = file.split("_")
        if len(parts) == 3:
            x, y, z_ext = parts
            new_name = f"{x}40C_{y}_{z_ext}"
        elif len(parts) == 4:
            additional_info, x, y, z_ext = parts
            new_name = f"{additional_info}_{x}40C_{y}_{z_ext}"
        else:   
            continue  # unexpected format
  
        shutil.copyfile(os.path.join(source_dir, file), os.path.join(renamed_dir, new_name))

        

In [ ]:
# Now we will rename them to figure_replicate.TIF format

# now we will rename them to figure_replicate.TIF format
# see length of json file counter

# see length of json file counter
with open(os.path.join(target_dir, "filename_mapping.json"), "r") as f:
    lines = f.readlines()
    last_mapping = json.loads(lines[-1])
    counter = int(last_mapping["target_filename"].split("_")[0]) + 1  # increment last count by 1

    
# Supp Fig 20

source_dir= renamed_dir

groups={}
json_data={}

for file in natsort.natsorted(os.listdir(source_dir)):
    if file.endswith(".TIF") and not file.startswith("._"):  # ignore leftover of Icloud backups
        parts = file.split("_")
        if len(parts) == 3:
            x, y, z_ext = parts
            additional_info = None
            group_key =x
        elif len(parts) == 4:
            additional_info, x, y, z_ext = parts
            group_key = f"{additional_info}_{x}"
        else:
            continue  # unexpected format

        z = z_ext.split(".")[0]  # remove .TIF extension

        # store in groups dictionary 
        if group_key not in groups:
            groups[group_key] = []
        groups[group_key].append((y, z))


for group_key, replicates in natsort.natsorted(groups.items()):
    
    for idx, (y, z) in enumerate(natsort.natsorted(replicates), start=1):
        source_filename = f"{group_key}_{y}_{z}.TIF" 
        target_filename = f"{counter}_{idx}.TIF"

        # copy and rename file
        shutil.copyfile(os.path.join(source_dir, source_filename), os.path.join(target_dir, target_filename))
        
        # store mapping in json
        json_data[target_filename] = source_filename

    counter+=1
      

# store json file 

with open(os.path.join(target_dir, "filename_mapping.json"), "a") as json_file:
    # json format is {source_filename: source_filename, target_filename: target_filename}
    for target_filename, source_filename in json_data.items():
        mapping = {"source_filename": source_filename, "target_filename": target_filename}
        json.dump(mapping, json_file)
        json_file.write("\n")  # write each mapping on a new line



In [14]:
# Supp Fig 24
# Two folders, we just add C1/C2 for biological replicate number, other than that looks fine
# Aggregate first too

renamed_dir = '/hpc/group/youlab/ks723/storage/Exp_images/41467_2025_65892_MOESM4_ESM/Source Data/SupFig24/Aggregated'
os.makedirs(renamed_dir, exist_ok=True)

# first replicate folder 
source_dir = '/hpc/group/youlab/ks723/storage/Exp_images/41467_2025_65892_MOESM4_ESM/Source Data/SupFig24/a/IndependentExperiment1_20250508'

for file in os.listdir(source_dir):
    if file.endswith(".TIF") and not file.startswith("._"):
        new_name = file

        # seperate to underscore parts 
        parts = new_name.split("_")
        
        if len(parts) == 3:
            additional_info, x, z_ext = parts
            new_name = f"{additional_info}_{x}_C1_{z_ext}"
        elif len(parts) == 2:
            x, z_ext = parts
            new_name = f"{x}_C1_{z_ext}"
        else:
            continue  # unexpected format

        shutil.copyfile(os.path.join(source_dir, file), os.path.join(renamed_dir, new_name))


# second replicate folder 

source_dir = '/hpc/group/youlab/ks723/storage/Exp_images/41467_2025_65892_MOESM4_ESM/Source Data/SupFig24/a/IndependentExperiment2_20250513'

for file in os.listdir(source_dir):
    if file.endswith(".TIF") and not file.startswith("._"):
        new_name = file

        # seperate to underscore parts 
        parts = new_name.split("_")
        
        if len(parts) == 3:
            additional_info, x, z_ext = parts
            new_name = f"{additional_info}_{x}_C2_{z_ext}"
        elif len(parts) == 2:
            x, z_ext = parts
            new_name = f"{x}_C2_{z_ext}"
        else:
            continue  # unexpected format

        shutil.copyfile(os.path.join(source_dir, file), os.path.join(renamed_dir, new_name))




In [15]:
# Now we will rename them to figure_replicate.TIF format

# now we will rename them to figure_replicate.TIF format
# see length of json file counter

# see length of json file counter
with open(os.path.join(target_dir, "filename_mapping.json"), "r") as f:
    lines = f.readlines()
    last_mapping = json.loads(lines[-1])
    counter = int(last_mapping["target_filename"].split("_")[0]) + 1  # increment last count by 1

    
# Supp Fig 24

source_dir= renamed_dir

groups={}
json_data={}

for file in natsort.natsorted(os.listdir(source_dir)):
    if file.endswith(".TIF") and not file.startswith("._"):  # ignore leftover of Icloud backups
        parts = file.split("_")
        if len(parts) == 3:
            x, y, z_ext = parts
            additional_info = None
            group_key =x
        elif len(parts) == 4:
            additional_info, x, y, z_ext = parts
            group_key = f"{additional_info}_{x}"
        else:
            continue  # unexpected format

        z = z_ext.split(".")[0]  # remove .TIF extension

        # store in groups dictionary 
        if group_key not in groups:
            groups[group_key] = []
        groups[group_key].append((y, z))


for group_key, replicates in natsort.natsorted(groups.items()):
    
    for idx, (y, z) in enumerate(natsort.natsorted(replicates), start=1):
        source_filename = f"{group_key}_{y}_{z}.TIF" 
        target_filename = f"{counter}_{idx}.TIF"

        # copy and rename file
        shutil.copyfile(os.path.join(source_dir, source_filename), os.path.join(target_dir, target_filename))
        
        # store mapping in json
        json_data[target_filename] = source_filename

    counter+=1
      

# store json file 

with open(os.path.join(target_dir, "filename_mapping.json"), "a") as json_file:
    # json format is {source_filename: source_filename, target_filename: target_filename}
    for target_filename, source_filename in json_data.items():
        mapping = {"source_filename": source_filename, "target_filename": target_filename}
        json.dump(mapping, json_file)
        json_file.write("\n")  # write each mapping on a new line



In [ ]:
# Supp Fig 25
# One folder, no technical replicates, can use directly 
# only thing is we have to change logic to two and three parts instead of three and four parts

renamed_dir = '/hpc/group/youlab/ks723/storage/Exp_images/41467_2025_65892_MOESM4_ESM/Source Data/SupFig25' 

# see length of json file counter
with open(os.path.join(target_dir, "filename_mapping.json"), "r") as f:
    lines = f.readlines()
    last_mapping = json.loads(lines[-1])
    counter = int(last_mapping["target_filename"].split("_")[0]) + 1  # increment last count by 1

    
# Supp Fig 25

source_dir= renamed_dir

groups={}
json_data={}

for file in natsort.natsorted(os.listdir(source_dir)):
    if file.endswith(".TIF") and not file.startswith("._"):  # ignore leftover of Icloud backups
        parts = file.split("_")
        if len(parts) == 2:
            x, z_ext = parts
            additional_info = None
            group_key =x
        elif len(parts) == 3:
            additional_info, x, z_ext = parts
            group_key = f"{additional_info}_{x}"
        else:
            continue  # unexpected format

        z = z_ext.split(".")[0]  # remove .TIF extension

        # store in groups dictionary 
        if group_key not in groups:
            groups[group_key] = []
        groups[group_key].append(z)


for group_key, replicates in natsort.natsorted(groups.items()):
    
    for idx, z in enumerate(natsort.natsorted(replicates), start=1):
        source_filename = f"{group_key}_{z}.TIF" 
        target_filename = f"{counter}_{idx}.TIF"

        # copy and rename file
        shutil.copyfile(os.path.join(source_dir, source_filename), os.path.join(target_dir, target_filename))
        
        # store mapping in json
        json_data[target_filename] = source_filename

    counter+=1
      

# store json file 

with open(os.path.join(target_dir, "filename_mapping.json"), "a") as json_file:
    # json format is {source_filename: source_filename, target_filename: target_filename}
    for target_filename, source_filename in json_data.items():
        mapping = {"source_filename": source_filename, "target_filename": target_filename}
        json.dump(mapping, json_file)
        json_file.write("\n")  # write each mapping on a new line


In [19]:
# Supp Fig 26 
# Looks like one folder is in paper, but missing from source data
# anyways lets proceed

renamed_dir = '/hpc/group/youlab/ks723/storage/Exp_images/41467_2025_65892_MOESM4_ESM/Source Data/SupFig26/Aggregated'
os.makedirs(renamed_dir, exist_ok=True)

# first replicate 
# ordering is weird, addtional info is second, we will shift to first 
# also there is an unneeded part at the start we will remove
source_dir = '/hpc/group/youlab/ks723/storage/Exp_images/41467_2025_65892_MOESM4_ESM/Source Data/SupFig26/IndependentReplicate1_TechnicalReplicate1/images'

for file in os.listdir(source_dir):
    if file.endswith(".TIF") and not file.startswith("._"):
        base, ext = os.path.splitext(file)
        parts = base.split("_")
        if len(parts) == 4:
            unneeded, x, additional_info, z = parts
            new_name = f"{additional_info}_{x}_C1_{z}{ext}"
        elif len(parts) == 3:
            unneeded, x, z = parts
            new_name = f"{x}_C1_{z}{ext}"
        else:
            continue  # unexpected format

        shutil.copyfile(os.path.join(source_dir, file), os.path.join(renamed_dir, new_name))

# second replicate 
# same as above
# plus one more change, change "C8minusBpE57" to "C8noBp"
 

source_dir = '/hpc/group/youlab/ks723/storage/Exp_images/41467_2025_65892_MOESM4_ESM/Source Data/SupFig26/IndependentReplicate2_TechnicalReplicate1/images'
for file in os.listdir(source_dir):
    if file.endswith(".TIF") and not file.startswith("._"):
        base, ext = os.path.splitext(file)
        parts = base.split("_")
        if len(parts) == 4:
            unneeded, x, additional_info, z = parts
            new_name = f"{additional_info}_{x}_C2_{z}{ext}"
        elif len(parts) == 3:
            unneeded, x, z = parts
            new_name = f"{x}_C2_{z}{ext}"
        else:
            continue  # unexpected format

        if "C8minusBpE57" in new_name:
            new_name = new_name.replace("C8minusBpE57", "C8noBp")

        shutil.copyfile(os.path.join(source_dir, file), os.path.join(renamed_dir, new_name))


# third replicate 
# same as above

source_dir = '/hpc/group/youlab/ks723/storage/Exp_images/41467_2025_65892_MOESM4_ESM/Source Data/SupFig26/IndependentReplicate2_TechnicalReplicate2/images'

for file in os.listdir(source_dir):
    if file.endswith(".TIF") and not file.startswith("._"):
        base, ext = os.path.splitext(file)
        parts = base.split("_")
        if len(parts) == 4:
            unneeded, x, additional_info, z = parts
            new_name = f"{additional_info}_{x}_C3_{z}{ext}"
        elif len(parts) == 3:
            unneeded, x, z = parts
            new_name = f"{x}_C3_{z}{ext}"
        else:
            continue  # unexpected format

        if "C8minusBpE57" in new_name:
            new_name = new_name.replace("C8minusBpE57", "C8noBp")

        shutil.copyfile(os.path.join(source_dir, file), os.path.join(renamed_dir, new_name))


In [20]:
# Now we will rename them to figure_replicate.TIF format

# now we will rename them to figure_replicate.TIF format
# see length of json file counter

# see length of json file counter
with open(os.path.join(target_dir, "filename_mapping.json"), "r") as f:
    lines = f.readlines()
    last_mapping = json.loads(lines[-1])
    counter = int(last_mapping["target_filename"].split("_")[0]) + 1  # increment last count by 1

    
# Supp Fig 26

source_dir= renamed_dir

groups={}
json_data={}

for file in natsort.natsorted(os.listdir(source_dir)):
    if file.endswith(".TIF") and not file.startswith("._"):  # ignore leftover of Icloud backups
        parts = file.split("_")
        if len(parts) == 3:
            x, y, z_ext = parts
            additional_info = None
            group_key =x
        elif len(parts) == 4:
            additional_info, x, y, z_ext = parts
            group_key = f"{additional_info}_{x}"
        else:
            continue  # unexpected format

        z = z_ext.split(".")[0]  # remove .TIF extension

        # store in groups dictionary 
        if group_key not in groups:
            groups[group_key] = []
        groups[group_key].append((y, z))


for group_key, replicates in natsort.natsorted(groups.items()):
    
    for idx, (y, z) in enumerate(natsort.natsorted(replicates), start=1):
        source_filename = f"{group_key}_{y}_{z}.TIF" 
        target_filename = f"{counter}_{idx}.TIF"

        # copy and rename file
        shutil.copyfile(os.path.join(source_dir, source_filename), os.path.join(target_dir, target_filename))
        
        # store mapping in json
        json_data[target_filename] = source_filename

    counter+=1
      

# store json file 

with open(os.path.join(target_dir, "filename_mapping.json"), "a") as json_file:
    # json format is {source_filename: source_filename, target_filename: target_filename}
    for target_filename, source_filename in json_data.items():
        mapping = {"source_filename": source_filename, "target_filename": target_filename}
        json.dump(mapping, json_file)
        json_file.write("\n")  # write each mapping on a new line

